In [ ]:
import re
import nltk
import time
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer



In [ ]:
# Functions

def preprocess_text(series):
     # @somone to @
    lst = []
    lstn = []
    length = len(series)
    for i, text in enumerate(series):
        # mark @, # and links
        text = ' '.join(['_at_someone_' if '@' in word else word for word in text.split()])
        text = ' '.join(['_link_' if 'http' in word or 'www.' in word else word for word in text.split()])
        text = ' '.join(['_hashtag_' if '#' in word else word for word in text.split()])
        # remove all other punctuation
        text = ''.join([word.lower() for word in text if word not in string.punctuation.replace('@','').replace('#','').replace('_','')])

        # the real work, pos tagging to reduce Nouns and other things that don't matter
        tokens = []
        nouns = []
        for a in nltk.pos_tag(text.split()):
            # keep these as-is for reasons
            if '_link_' in a[0] or '_at_someone_' in a[0] or '_hashtag_' in a[0]:
                tokens.append(a[0])
                nouns.append(a[0])
            # lemmatize these to reduce parts of speach issues
            elif 'NN' in a[1] or 'JJ' in a[1] or 'RB' in a[1] or 'V' in a[1]:
                tokens.append(a[1])
                try:
                    nouns.append(nltk.PorterStemmer().stem(a[0]))
                except:
                    print('Recursion error',a)
            else:
                tokens.append(a[1])
                #nouns.append(nltk.PorterStemmer().stem(a[0]))
        lst.append(' '.join(tokens))
        lstn.append(' '.join(nouns))
        if i%100 == 0:
            print("\r{:.2f}% Complete".format(100*((i+1)/length), a), end="", flush=True)
    print("\r100.00% Complete".format(100*((i+1)/length)), end="", flush=True)
    return lst, lstn

def vectorize_data(df, nounVectorizer, gramVectorizer, charVectorizer, scaler):
    gV = gramVectorizer.transform(df['pos'])
    nV = nounVectorizer.transform(df['stems'])
    nC = charVectorizer.transform(df['comment_text'])
    transformed_data = hstack([gV, nV, nC])

    if scaler != None:
        scaler.fit(transformed_data)
    return transformed_data

def fit_vectorizers(df):
    # gram is based on the work I did in howRussian
    gramVectorizer =  TfidfVectorizer(ngram_range=(2,5)
                                        , min_df = 10
                                        , max_df = .7
                                        , stop_words = None
                                        , lowercase=False              # Protect POS tag case sensitivity
                                        , token_pattern=r'\S+'          # Tokenize strictly by space-separated tags/tokens
                                       )
    gramVectorizer.fit(df['pos'])
    
    
    
    # a simple vectorizer on the nouns to include specific words because reasons, ideal at word and bigram
    nounVectorizer =  TfidfVectorizer(ngram_range=(1,3)
                                        , min_df = 5
                                        , max_df = 0.75
                                        , stop_words = nltk.corpus.stopwords.words('english')
                                       )
    nounVectorizer.fit(df['stems'])


    # char_vec to catch weird mispellings...
    charVectorizer = TfidfVectorizer(ngram_range=(3, 5)
                                        , analyzer='char'
                                        , sublinear_tf=True
                                        , min_df=5
                                        , max_df = 0.5
                                        , max_features=50000
                                       )
    charVectorizer.fit(df['comment_text'])

    
    gV = gramVectorizer.transform(df['pos'])
    nV = nounVectorizer.transform(df['stems'])
    nC = charVectorizer.transform(df['comment_text'])
    transformed_data = hstack([gV, nV, nC])

    scaler = StandardScaler(with_mean=False)
    # scaler = None
    scaler.fit(transformed_data)
    return gramVectorizer, nounVectorizer, charVectorizer, scaler

def set_threshold(model = None, models = None, col = None, df=None, Xt=None):
    testing_gate = pd.DataFrame()
    testing_gate['id'] = df['id']
    cols = ['ttoxic', 'tobscene', 'tthreat', 'tinsult', 'tidentity_hate']
    if col != None and col != 'flag':
        for i, model in enumerate(models):
             testing_gate[cols[i]] = model.predict_proba(Xt)[:, 1]
    else:
        testing_gate['tflag'] = model.predict_proba(Xt)[:, 1]
    
    ndf = df.copy().merge(testing_gate, on='id')

    cols = ['toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    if col == None:
        col = 'flag'
    if model == None:
        model = models[cols.index(col)]

    if col == 'flag':
        # Get false positive rates, true positive rates, and thresholds
        fpr, tpr, thresholds = roc_curve(ndf[col], ndf['t'+col])
        j_scores = tpr - fpr
        best_idx = j_scores.argmax()
        best_threshold = thresholds[best_idx]
        return best_threshold
    else:
        precision, recall, thresholds = precision_recall_curve(ndf[col], ndf['t' + col])
        
        # Calculate F1 score for each threshold
        # (Adding 1e-10 prevents division by zero errors)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        
        # Find the index of the highest F1 score
        best_idx = f1_scores.argmax()
        best_threshold = thresholds[best_idx]
        return best_threshold
    
def test_models(gatekeeper, models, gthresh, mthresh, ndf, Xt): 
    # need the percents
    ndf['tflag'] = gatekeeper.predict_proba(Xt)[:, 1]
    cols = ['ttoxic','tobscene', 'tthreat', 'tinsult', 'tidentity_hate']
    for i, model in enumerate(models):
         ndf[cols[i]] = model.predict_proba(Xt)[:, 1]
    cols = ['toxic', 'obscene', 'threat', 'insult', 'identity_hate']
   
    false_positives = 0
    false_negatives = 0
    true_positives = 0
    true_negatives = 0
    
    for i, row in ndf.iterrows():
        if row['flag'] == 0 and row['tflag'] < gthresh:
            true_negatives +=1
        elif row['flag'] == 0 and row['tflag'] >= gthresh:
            false_positives += 1
        elif row['flag'] == 1 and row['tflag'] < gthresh:
            false_negatives += 1
        else:
            checkfn = 0
            checkfp = 0
            for col in cols:
                if row[col] == 0 and row['t'+col] >= mthresh[cols.index(col)]:
                    checkfp += 1
                elif row[col] == 1 and row['t'+col] <  mthresh[cols.index(col)]:
                    checkfn += 1
            if checkfn >= 1:
                false_negatives += 1
    
            if checkfp >= 1:
                false_positives += 1
            if checkfn == 0 and checkfp == 0:
                true_positives += 1
  
    print('TP: {} TN: {} FP: {} FN: {}'.format(true_positives, true_negatives, false_positives, false_negatives))

def train_model(Xtrain, Xt, train_df, test_df, col, verbose=True):
    
    y = train_df[col]
    yt = test_df[col]
    
    model = LogisticRegression(
        C = 0.5
        , penalty='l2'
        , solver='sag'
        , class_weight='balanced'
        , n_jobs = -1
    )
    
    model.fit(Xtrain,y)
    
    if verbose:
        print('Training Data {:.3f}, Test Data Score: {:.3f}'.format(model.score(Xtrain,y), model.score(Xt,yt)))
        y_pred_probs = model.predict_proba(Xtrain)[:, 1]
        yt_pred_probs = model.predict_proba(Xt)[:, 1]
        # Calculate ROC-AUC for the single binary label
        auc_score = roc_auc_score(y, y_pred_probs)
        
        auc_scoret = roc_auc_score(yt, yt_pred_probs)
        print(f"TRAINING ROC-AUC Score: {auc_score:.4f}, Test ROC-AUC Score: {auc_scoret:.4f}")
    return model

def build_model(model_score = 0.0, ogatekeeper = None, omodels = None, ogatekeeper_thresh = 0, othresholds = [0,0,0,0,0,0]):

    stopword = nltk.corpus.stopwords.words('english')
    ps = nltk.PorterStemmer()

    # get and process data
    df = pd.read_csv('data/processed_data.csv')
    df['pos'], df['stems'] = preprocess_text(df['comment_text'])
    
    # split the data
    train_df, test_df = train_test_split(df, test_size=0.2)
    gramVectorizer, nounVectorizer, charVectorizer, scaler = fit_vectorizers(train_df)
    
    # flag_df
    temp1 = train_df[train_df['flag'] == 1]
    # get the last 25000 comemnts in each group, focus on newer neg comments
    temp_ = pd.DataFrame()
    for col in ['toxic', 'obscene', 'threat', 'insult', 'identity_hate']:
        temp_ = pd.concat([temp_, temp1[temp1[col]==1].tail(25000)])
    print('temp_', temp_.shape)
    temp1 = temp_.drop_duplicates()
    print('temp1', temp1.shape)
   
    # 
    temp2 = train_df[train_df['flag'] == 0].sort_values('pred', ascending=False).head(temp1.shape[0])
    flag_df = pd.concat([temp1, temp2]).sort_index()
    tox_df = temp1
    test_flag_df = test_df[test_df['flag']==1]

    # vectorize!
    Xf = vectorize_data(flag_df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
    Xt = vectorize_data(test_df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
    Xx = vectorize_data(tox_df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
    Xtf = vectorize_data(test_flag_df, nounVectorizer, gramVectorizer, charVectorizer, scaler)

    # train the models
    gatekeeper = train_model(Xf, Xt, flag_df, test_df, col='flag', verbose=True)
    models = []
    models.append(train_model(Xf, Xtf, flag_df, test_flag_df, col='toxic', verbose=True))  # otherwise all toxic
    for col in ['obscene', 'threat', 'insult', 'identity_hate']:
        models.append(train_model(Xx, Xtf, tox_df, test_flag_df, col='toxic', verbose=True))
        
    gatekeeper_thresh = set_threshold(model = gatekeeper, models = None, col = 'flag', df=flag_df, Xt=Xf)
    thresholds = []
    thresholds.append(set_threshold(model = None, models = models, col = 'toxic', df=flag_df, Xt=Xf))
    [thresholds.append(set_threshold(model = None, models = models, col = col, df=flag_df, Xt=Xf)) for col in ['obscene', 'threat', 'insult', 'identity_hate']]
    test_models(gatekeeper, models, gatekeeper_thresh, thresholds, ndf=test_df, Xt=Xt)

    # check score...
    scores  = []
    yt = test_df['flag']
    gscore = gatekeeper.score(Xt,yt)
    for i, col in enumerate(['toxic', 'obscene', 'threat', 'insult', 'identity_hate']):
        yt = test_df[col]
        scores.append(models[i].score(Xt, yt))
    mscore = sum(scores)/len(scores)
    # gatekeeper is more important...
    score = (gscore+mscore)/2
    print('newscore', score)
    if score >= model_score:  # = so newer models are preferred...
        #redo preds for current model
        X = vectorize_data(df, nounVectorizer, gramVectorizer, charVectorizer, scaler)
        df['pred'] = gatekeeper.predict_proba(X)[:, 1]
        df.to_csv('data/processed_data.csv', index=False)
        return score, gatekeeper, models, gatekeeper_thresh, thresholds
    else:
        return model_score, ogatekeeper, omodels, ogatekeeper_thresh, othresholds
    

In [ ]:
model_score, gatekeeper, models, gatekeeper_thresh, thresholds = build_model()
print()
print(model_score)

In [ ]:
model_score, gatekeeper, models, gatekeeper_thresh, thresholds = build_model(model_score, gatekeeper, models, gatekeeper_thresh, thresholds)
print()
print(model_score)

In [1]:
import sqlite3

# Connects to the database (creates 'company.db' if it doesn't exist)
connection = sqlite3.connect("company.db")

# Create a cursor object to execute SQL commands
cursor = connection.cursor()

# Create a sample table
cursor.execute("""
CREATE TABLE posts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pet_id TEXT NOT NULL,
    platform TEXT NOT NULL,
    post_id TEXT NOT NULL UNIQUE,
    post_url TEXT NOT NULL,
    posted_at TEXT NOT NULL,
    created_at TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at TEXT NOT NULL DEFAULT (datetime('now'))
);
""")
CREATE TABLE post_metrics (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    post_id TEXT NOT NULL,
    collected_at TEXT NOT NULL,
    likes INTEGER,
    reposts INTEGER,
    comments INTEGER,
    created_at TEXT NOT NULL DEFAULT (datetime('now')),
    updated_at TEXT NOT NULL DEFAULT (datetime('now')),
    FOREIGN KEY (post_id) REFERENCES posts (post_id)
);
""")

# Commit changes and close the connection
connection.commit()
connection.close()

print("Database and table created successfully!")

ProgrammingError: You can only execute one statement at a time.